In [1]:
# Notebook: stellar-8model-lr-stacker
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
import gc, warnings
warnings.filterwarnings('ignore')

MODEL_PATH = '/kaggle/input/datasets/anjanamohan13/stellar-model-outputs/'
TRAIN_CSV  = '/kaggle/input/competitions/playground-series-s6e6/train.csv'
TEST_CSV   = '/kaggle/input/competitions/playground-series-s6e6/test.csv'
SAMPLE_SUB = '/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv'

N_FOLDS = 5
N_SEEDS = 5
SEEDS   = list(range(42, 42 + N_SEEDS))
epochs  = 1000
C       = 0.1
EPS     = 1e-15
LOGIT_CLIP = 30.0

TARGET     = 'class'
target_map = {'GALAXY': 0, 'QSO': 1, 'STAR': 2}
inv_map    = {v: k for k, v in target_map.items()}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Seeds: {SEEDS}')

Device: cuda
Seeds: [42, 43, 44, 45, 46]


In [2]:
train = pd.read_csv(TRAIN_CSV)
test  = pd.read_csv(TEST_CSV)
y = train[TARGET].map(target_map).astype(int).values
N = len(y)
M = len(test)
print(f'Train: {N:,} | Test: {M:,}')

Train: 577,347 | Test: 247,435


In [3]:
import os
print(os.listdir(MODEL_PATH))

['test_xgb_v2.csv', 'oof_xgb_v2.csv', 'test_xgb2.csv', 'oof_xgb2.csv', 'oof_mlp (3).csv', 'test_mlp3.csv', 'oof_lgb (2).csv', 'oof_mlp2.csv', 'test_tabm.csv', 'test_mlp2.csv', 'test_xgb.csv', 'test_cat.csv', 'oof_cat.npy', 'oof_xgb.csv', 'oof_lgb2.csv', 'test_lgb2.csv', 'test_mlp.csv', 'test_mlp (1).csv', 'oof_stacker5.npy', 'test_lgb (1).csv', 'oof_mlp (2).csv', 'oof_mlp3.csv', 'oof_tabm.csv', 'oof_lgb (3).csv', 'test_lgb.csv', 'test_cat.npy', 'test_stacker5.npy', 'oof_cat.csv']


In [4]:
STACKING_FILES = [
    ('realmlp',  MODEL_PATH + 'oof_mlp (3).csv', MODEL_PATH + 'test_mlp (1).csv'),
    ('realmlp2', MODEL_PATH + 'oof_mlp2.csv',    MODEL_PATH + 'test_mlp2.csv'),
    ('realmlp3', MODEL_PATH + 'oof_mlp3.csv',    MODEL_PATH + 'test_mlp3.csv'),
    ('lgb',      MODEL_PATH + 'oof_lgb (3).csv', MODEL_PATH + 'test_lgb (1).csv'),
    ('xgb',      MODEL_PATH + 'oof_xgb.csv',     MODEL_PATH + 'test_xgb.csv'),
    ('xgb_v2',   MODEL_PATH + 'oof_xgb_v2.csv',  MODEL_PATH + 'test_xgb_v2.csv'),
    ('cat',      MODEL_PATH + 'oof_cat.csv',     MODEL_PATH + 'test_cat.csv'),
    ('cat2',     MODEL_PATH + 'oof_cat2.csv',    MODEL_PATH + 'test_cat2.csv'),
]

def prob_to_logit(p):
    p = np.clip(p, EPS, 1.0 - EPS).astype(np.float64)
    return np.clip(np.log(p / (1.0 - p)), -LOGIT_CLIP, LOGIT_CLIP).astype(np.float32)

def load_preds(path, expected_rows):
    df = pd.read_csv(path)
    if df.shape[1] == 1:
        vals = df.iloc[:, 0].values
        assert len(vals) == expected_rows * 3
        return vals.reshape(-1, 3)
    return df.iloc[:, -3:].values[:expected_rows]

loaded_oofs, loaded_tests, names = [], [], []
print('Loading models:')
for name, oof_f, test_f in STACKING_FILES:
    try:
        o = prob_to_logit(load_preds(oof_f, N))
        t = prob_to_logit(load_preds(test_f, M))
        assert o.shape == (N, 3)
        assert t.shape == (M, 3)
        loaded_oofs.append(o)
        loaded_tests.append(t)
        names.append(name)
        print(f'  {name:15s} OOF={o.shape} TEST={t.shape}')
    except Exception as e:
        print(f'  SKIP {name}: {e}')

n_models = len(loaded_oofs)
print(f'\nModels loaded: {n_models} | LR input features: {n_models * 3}')

X_test_tensor = torch.tensor(
    np.concatenate(loaded_tests, axis=1).astype(np.float32),
    dtype=torch.float32, device=device
)

Loading models:
  realmlp         OOF=(577347, 3) TEST=(247435, 3)
  realmlp2        OOF=(577347, 3) TEST=(247435, 3)
  realmlp3        OOF=(577347, 3) TEST=(247435, 3)
  lgb             OOF=(577347, 3) TEST=(247435, 3)
  xgb             OOF=(577347, 3) TEST=(247435, 3)
  xgb_v2          OOF=(577347, 3) TEST=(247435, 3)
  cat             OOF=(577347, 3) TEST=(247435, 3)
  SKIP cat2: [Errno 2] No such file or directory: '/kaggle/input/datasets/anjanamohan13/stellar-model-outputs/oof_cat2.csv'

Models loaded: 7 | LR input features: 21


In [5]:
class MultiLogReg(nn.Module):
    def __init__(self, in_features, out_features=3):
        super().__init__()
        self.linear = nn.Linear(in_features, out_features)
    def forward(self, x):
        return self.linear(x)

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

oof_sum  = np.zeros((N, 3), dtype=np.float64)
test_sum = np.zeros((M, 3), dtype=np.float64)
all_scores = []

for seed in SEEDS:
    set_seed(seed)
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof_this  = np.zeros((N, 3), dtype=np.float32)
    test_this = np.zeros((M, 3), dtype=np.float32)
    seed_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(np.zeros(N), y), 1):
        X_tr  = np.concatenate([o[tr_idx]  for o in loaded_oofs], axis=1).astype(np.float32)
        X_val = np.concatenate([o[val_idx] for o in loaded_oofs], axis=1).astype(np.float32)
        y_tr  = y[tr_idx]
        y_val = y[val_idx]

        classes = np.unique(y_tr)
        cw = compute_class_weight('balanced', classes=classes, y=y_tr)
        sw = np.array([dict(zip(classes, cw))[c] for c in y_tr], dtype=np.float32)

        X_tr_t  = torch.tensor(X_tr,  dtype=torch.float32, device=device)
        y_tr_t  = torch.tensor(y_tr,  dtype=torch.long,    device=device)
        sw_t    = torch.tensor(sw,     dtype=torch.float32, device=device)
        X_val_t = torch.tensor(X_val, dtype=torch.float32, device=device)

        model = MultiLogReg(X_tr.shape[1]).to(device)
        wd    = 1.0 / (C * len(y_tr))
        optimizer = optim.Adam([
            {'params': model.linear.weight, 'weight_decay': wd},
            {'params': model.linear.bias,   'weight_decay': 0.0},
        ], lr=0.01)
        criterion = nn.CrossEntropyLoss(reduction='none')

        model.train()
        for _ in range(epochs):
            optimizer.zero_grad()
            loss = (criterion(model(X_tr_t), y_tr_t) * sw_t).mean()
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_prob  = torch.softmax(model(X_val_t), dim=1).cpu().numpy()
            test_prob = torch.softmax(model(X_test_tensor), dim=1).cpu().numpy()

        oof_this[val_idx] = val_prob
        test_this += test_prob / N_FOLDS

        score = balanced_accuracy_score(y_val, np.argmax(val_prob, axis=1))
        seed_scores.append(score)

        del model, X_tr_t, y_tr_t, sw_t, X_val_t
        gc.collect()

    oof_sum  += oof_this
    test_sum += test_this

    seed_bac = balanced_accuracy_score(y, np.argmax(oof_this, axis=1))
    all_scores.extend(seed_scores)
    print(f'Seed {seed}: OOF={seed_bac:.5f} | folds={[f"{s:.5f}" for s in seed_scores]}')

oof        = (oof_sum  / N_SEEDS).astype(np.float32)
test_preds = (test_sum / N_SEEDS).astype(np.float32)

overall = balanced_accuracy_score(y, np.argmax(oof, axis=1))
print(f'\nMean fold score: {np.mean(all_scores):.5f} ± {np.std(all_scores):.5f}')
print(f'Overall OOF BAC (5-seed avg): {overall:.5f}')

Seed 42: OOF=0.96894 | folds=['0.96993', '0.96889', '0.96822', '0.96874', '0.96893']
Seed 43: OOF=0.96895 | folds=['0.96861', '0.96982', '0.96970', '0.96815', '0.96849']
Seed 44: OOF=0.96910 | folds=['0.96899', '0.96940', '0.96925', '0.96875', '0.96914']
Seed 45: OOF=0.96894 | folds=['0.96876', '0.96862', '0.96921', '0.96927', '0.96885']
Seed 46: OOF=0.96890 | folds=['0.96901', '0.96927', '0.96954', '0.96875', '0.96792']

Mean fold score: 0.96897 ± 0.00049
Overall OOF BAC (5-seed avg): 0.96904


In [6]:
sub = pd.read_csv(SAMPLE_SUB)
sub[TARGET] = np.argmax(test_preds, axis=1)
sub[TARGET] = sub[TARGET].map(inv_map)
sub.to_csv('submission_8model_stacked.csv', index=False)

np.save('oof_stacker8.npy',  oof)
np.save('test_stacker8.npy', test_preds)

print('✅ submission_8model_stacked.csv saved!')
print(sub[TARGET].value_counts())
sub.head(5)

✅ submission_8model_stacked.csv saved!
class
GALAXY    156856
QSO        51439
STAR       39140
Name: count, dtype: int64


,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
